In [2]:
import pandas as pd
import numpy as np
import torch
from modules import MLPDiffusion
from gaussian_multinomial_diffsuion import GaussianMultinomialDiffusion
feature_cols = ["Tau_c","Tau_p",
    "RSSC Vel","CAV","CAD",
    "ID_2","IV_2","Arias intensity UD","Arias intensity NS","Arias intensity EW",
    "PGV UD 3 sec","PGD UD 3 sec","PGA UD 3 sec",
    "Station_Lat","Station_Lon","Focal depth",
    "Epicenter_Lat","V VS ACC","Vs30",
    "Epicenter_Lon","Total PGV UD","Total PGD UD","Total PGV NS","Total PGD NS","Total PGV EW","Total PGD EW",
    "Magnitude"]
base = r"D:\Japan Dataset\Input Dataset"
train = pd.read_csv(rf"{base}\Train_tab_MAG_data_3_SEC.csv")
val   = pd.read_csv(rf"{base}\Test_tab_MAG_data_3_SEC.csv")
real = pd.concat([train, val], ignore_index=True)
real = real[feature_cols]
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(real)
X = torch.tensor(X, dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X = X.to(device)
model = MLPDiffusion(
    d_in=X.shape[1],
    num_classes=0,
    is_y_cond=False,
    rtdl_params={
        "d_layers": [512, 256, 256],
        "dropout": 0.1
    }).to(device)
diffusion = GaussianMultinomialDiffusion(
    num_classes=np.array([0]),      # one dummy entry meaning no categorical columns
    num_numerical_features=X.shape[1],
    denoise_fn=model,
    num_timesteps=20,
    gaussian_loss_type="mse",
    gaussian_parametrization="eps",
    scheduler="cosine",
    device=device).to(device)
optimizer = torch.optim.Adam(diffusion.parameters(),lr=0.0012)
loader = torch.utils.data.DataLoader(
    X,
    batch_size=256,
    shuffle=True
)
epochs = 50
for epoch in range(epochs):
    diffusion.train()
    running_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss_multi, loss_gauss = diffusion.mixed_loss(
            batch,
            out_dict={})
        loss = loss_multi + loss_gauss
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(
        f"Epoch {epoch+1:4d} | Loss = {running_loss/len(loader):.6f}")
print("Training Complete.")

Epoch    1 | Loss = 0.887242
Epoch    2 | Loss = 0.615259
Epoch    3 | Loss = 0.494110
Epoch    4 | Loss = 0.449044
Epoch    5 | Loss = 0.413962
Epoch    6 | Loss = 0.404711
Epoch    7 | Loss = 0.388296
Epoch    8 | Loss = 0.384042
Epoch    9 | Loss = 0.367383
Epoch   10 | Loss = 0.373497
Epoch   11 | Loss = 0.354856
Epoch   12 | Loss = 0.342901
Epoch   13 | Loss = 0.332812
Epoch   14 | Loss = 0.332154
Epoch   15 | Loss = 0.326790
Epoch   16 | Loss = 0.336694
Epoch   17 | Loss = 0.324978
Epoch   18 | Loss = 0.320031
Epoch   19 | Loss = 0.313161
Epoch   20 | Loss = 0.307859
Epoch   21 | Loss = 0.328169
Epoch   22 | Loss = 0.305443
Epoch   23 | Loss = 0.307755
Epoch   24 | Loss = 0.312901
Epoch   25 | Loss = 0.310613
Epoch   26 | Loss = 0.297235
Epoch   27 | Loss = 0.315640
Epoch   28 | Loss = 0.302758
Epoch   29 | Loss = 0.298750
Epoch   30 | Loss = 0.296042
Epoch   31 | Loss = 0.301580
Epoch   32 | Loss = 0.297801
Epoch   33 | Loss = 0.302943
Epoch   34 | Loss = 0.296951
Epoch   35 | L

In [2]:
n_samples = 180000
batch_size = 1000
synthetic_data = []
import numpy as np
import numpy as np
import pandas as pd
def generate(train_df,batch_size=256):
    noise = torch.randn(
            current_batch,
            train_df.shape[1],
            device=device)
    # DDIM Sampling
    samples = diffusion.gaussian_ddim_sample(
            noise=noise,
            T=10,
            out_dict={},
            eta=0.1)
    samples = samples.cpu().numpy()
    # Undo StandardScaler
    samples = scaler.inverse_transform(samples)
    X = pd.DataFrame(samples, columns=feature_cols)
    X=X[X['Tau_c']>0]
    X=X[X['TP']>0]
    X=X[X['RSSC Vel']>0]
    X=X[X['CAV']>0]
    X=X[X['CAD']>0]
    X=X[X['ID_2']>0]
    X=X[X['IV_2']>0]
    X=X[X['Arias intensity UD']>0]
    X=X[X['Arias intensity NS']>0]
    X=X[X['Arias intensity EW']>0]
    X=X[X['PGD UD 3 sec']>0]
    X=X[X['PGV UD 3 sec']>0]
    X=X[X['PGA UD 3 sec']>0]
    X=X[X['Total PGD UD']>0]
    X=X[X['Total PGA UD']>0]
    X=X[X['Total PGV UD']>0]
    X=X[X['Total PGD EW']>0]
    X=X[X['Total PGA EW']>0]
    X=X[X['Total PGV EW']>0]
    X=X[X['Total PGD NS']>0]
    X=X[X['Total PGA NS']>0]
    X=X[X['Total PGV NS']>0]
    X=X[X['Focal depth']>0]
    X=X[X['Vs30']>0]
    min_st_lon=np.min(train_df['Station_Lon'])-1
    max_st_lon=np.max(train_df['Station_Lon'])+1
    min_st_lat=np.min(train_df['Station_Lat'])-1
    max_st_lat=np.max(train_df['Station_Lat'])+1
    X=X[X['Station_Lat']>=min_st_lat]
    X=X[X['Station_Lat']<=max_st_lat]
    X=X[X['Station_Lon']>=min_st_lon]
    X=X[X['Station_Lon']<=max_st_lon]
    return X
synthetic_data = []
train_file = r"D:\Japan Dataset\Input Dataset\Test_tab_MAG_data_3_SEC.csv"
feature_cols = ["Tau_c","TP","RSSC Vel","CAV","CAD","ID_2","IV_2","Arias intensity UD",
        "PGD UD 3 sec","Station_Lat","Station_Lon","Depth_km","Epicenter_Lat","Epicenter_Lon","Magnitude"]
    # Read only required columns
train_df = pd.read_csv(train_file, usecols=feature_cols)
for i in range(0, n_samples, batch_size):
    current_batch = min(batch_size, n_samples - i)
    samples = generate(train_df,batch_size=current_batch)
    synthetic_data.append(samples)

synthetic_data = np.vstack(synthetic_data)
synthetic_df = pd.DataFrame(synthetic_data, columns=feature_cols)
output_file = r"D:\Japan Dataset\Output Dataset\Synthetic_India_Train_Tabddpm.csv"

synthetic_df.to_csv(output_file, index=False)
print("\n=====================================")
print("Synthetic Dataset Generation Finished")
print("Shape :", synthetic_df.shape)
print("Saved :", output_file)
print("=====================================")

Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timest